# 33 — EXP_28: Official + Cleaned Open Pool Multi-View LinearSVC
EXPERIMENT_ID: 28_open_pool_svc
DATE: 2026-11-06
RESEARCH_QUESTION: Does adding 36,000 clean, balanced, external Open Pool passages
                   (2,000 per class across all 18 models, sample-weighted at 0.35)
                   improve prompt-disjoint balanced accuracy over official-only 4-view SVC (0.3749)?
HYPOTHESIS:
1. The official training set contains only ~150 passages per model (2,699 total), leading to sparse n-gram representations.
2. The Open Pool provides 36,000 verified passages generated by the 18 candidate models across thousands of diverse open prompts.
3. Incorporating these external rows with source weight 0.35 and strict prompt contamination filtering
   will expand the 4-view vocabulary and substantially boost generalization on unseen validation prompts.
DATA_VERSION: Official v1 (2,699) + Clean Open Pool (36,000 balanced)
CV_STRATEGY: GroupKFold(n_splits=5, group=prompt_id) (Strict Prompt-Disjoint on Official Data)
OUTPUTS: outputs/oof_open_pool_svc.npy, reports/open_pool_svc_results.md

In [ ]:
import os, sys, time, json, datetime, re
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from scipy.special import softmax
from sklearn.model_selection import GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import balanced_accuracy_score, f1_score, classification_report

BASE_DIR = "/home/rj2826/name-the-llm"
DATA_PATH = os.path.join(BASE_DIR, "data/train.csv")
OPEN_POOL_PATH = os.path.join(BASE_DIR, "data/external/open_pool_clean.parquet")
OUTPUTS_DIR = os.path.join(BASE_DIR, "outputs")
REPORTS_DIR = os.path.join(BASE_DIR, "reports")
EXPERIMENTS_DIR = os.path.join(BASE_DIR, "experiments")
os.makedirs(REPORTS_DIR, exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)

In [ ]:
print("Loading official competition dataset...")
train = pd.read_csv(DATA_PATH)
texts_off = train["text"].fillna("").astype(str).values
y_true = train["model"].values
groups = train["prompt_id"].values
classes = sorted(np.unique(y_true))
n_classes = len(classes)
class_to_idx = {c: i for i, c in enumerate(classes)}
y_true_idx = np.array([class_to_idx[c] for c in y_true])

In [ ]:
print("Loading clean Open Pool dataset...")
pool_df = pd.read_parquet(OPEN_POOL_PATH)
print(f"Total clean pool rows: {len(pool_df):,}")

# Sample balanced 2,000 rows per class (36,000 rows total)
pool_balanced = pd.concat(
    [g.sample(min(len(g), 2000), random_state=42) for _, g in pool_df.groupby("model")],
    ignore_index=True
)
print(f"Balanced external pool: {len(pool_balanced):,} rows (2,000 per class across 18 models)")

texts_ext = pool_balanced["text"].fillna("").astype(str).values
y_ext_idx = pool_balanced["model"].map(class_to_idx).to_numpy(dtype=int)

In [ ]:
FW_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())

def shape_transform(text):
    return re.sub(r"[^\W\d_]+", lambda m: "X" if m.group(0)[0].isupper() else "x", text)

def func_transform(text):
    tokens = re.findall(r"\w+|[^\w\s]", text)
    transformed = [w.lower() if w.lower() in FW_WORDS or not w.isalnum() else "CONTENT" for w in tokens]
    return " ".join(transformed)

In [ ]:
kf = GroupKFold(n_splits=5)
oof_probs = np.zeros((len(train), n_classes), dtype=np.float32)
oof_preds = np.zeros(len(train), dtype=int)
fold_scores = []
contamination_stats = []

views_config = [
    ("char", 3, 5, 120000, 1.0),
    ("word", 1, 3, 80000, 0.7),
    ("shape", 2, 6, 40000, 0.5),
    ("function", 1, 3, 40000, 0.5),
]

t0 = time.time()
print("\n" + "=" * 75)
print("EXECUTING EXP_28: OFFICIAL + CLEANED OPEN POOL 4-VIEW LINEARSVC")
print("=" * 75)

for fold, (tr_idx, val_idx) in enumerate(kf.split(train, y_true, groups=groups), 1):
    f_t0 = time.time()
    
    # Official fold partitions
    tr_off_texts = texts_off[tr_idx]
    val_off_texts = texts_off[val_idx]
    y_tr_off = y_true_idx[tr_idx]
    y_val_off = y_true_idx[val_idx]
    val_prompts = set(groups[val_idx])
    
    # Validation Contamination Check:
    # Ensure no external row has high TF-IDF similarity to the validation prompts of this fold
    vec_check = TfidfVectorizer(stop_words='english', min_df=2, max_df=0.2)
    val_prompt_text = [" ".join(texts_off[groups == p]) for p in val_prompts]
    V_val = vec_check.fit_transform(val_prompt_text)
    V_ext = vec_check.transform(texts_ext)
    sim_to_val = (V_ext @ V_val.T).max(axis=1).toarray().ravel()
    
    # Mask out any external row with cosine similarity > 0.65 to this fold's validation prompts
    contam_mask = sim_to_val > 0.65
    purged_count = int(np.sum(contam_mask))
    contamination_stats.append(purged_count)
    
    ext_keep = ~contam_mask
    tr_ext_texts = texts_ext[ext_keep]
    y_tr_ext = y_ext_idx[ext_keep]
    
    # Combine training sets
    tr_all_texts = np.concatenate([tr_off_texts, tr_ext_texts])
    y_tr_all = np.concatenate([y_tr_off, y_tr_ext])
    
    # Sample weights: official = 1.0, external pool = 0.35 (matching reference pipeline)
    sw_all = np.concatenate([np.ones(len(tr_off_texts), dtype=np.float32), np.full(len(tr_ext_texts), 0.35, dtype=np.float32)])
    
    # Feature Extraction
    A_tr, B_val = [], []
    for view, lo, hi, mf, wv in views_config:
        prep = shape_transform if view == "shape" else func_transform if view == "function" else (lambda s: s)
        an = "char" if view == "shape" else "word" if view == "function" else view
        
        o = dict(analyzer=an, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=3, max_features=mf, dtype=np.float32)
        if an == "word":
            o["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
            
        vec = TfidfVectorizer(**o)
        tr_prep = [prep(t) for t in tr_all_texts]
        val_prep = [prep(t) for t in val_off_texts]
        
        X_tr_v = vec.fit_transform(tr_prep) * wv
        X_val_v = vec.transform(val_prep) * wv
        A_tr.append(X_tr_v)
        B_val.append(X_val_v)
        
    X_tr_all = hstack(A_tr, format="csr")
    X_val_all = hstack(B_val, format="csr")
    
    # Train LinearSVC with balanced class weights and sample weights
    clf = LinearSVC(C=0.5, class_weight="balanced", random_state=42, max_iter=3000)
    clf.fit(X_tr_all, y_tr_all, sample_weight=sw_all)
    
    d = clf.decision_function(X_val_all)
    val_p = softmax(d, axis=1)
    oof_probs[val_idx] = val_p
    preds = np.argmax(val_p, axis=1)
    oof_preds[val_idx] = preds
    
    f_acc = balanced_accuracy_score(y_val_off, preds)
    fold_scores.append(f_acc)
    print(f"Fold {fold} in {time.time() - f_t0:.1f}s | BalAcc={f_acc:.4f} | TrRows={len(tr_all_texts)} (Ext: {len(tr_ext_texts)}, Purged: {purged_count}) | Features={X_tr_all.shape[1]}")

total_elapsed = time.time() - t0
mean_acc = balanced_accuracy_score(y_true_idx, oof_preds)
macro_f1 = f1_score(y_true_idx, oof_preds, average="macro")

print("\n" + "=" * 75)
print(f"EXP_28 OPEN POOL SVC RESULTS:")
print(f"Mean Balanced Accuracy: {mean_acc:.4f} (±{np.std(fold_scores):.4f})")
print(f"Macro F1 Score:         {macro_f1:.4f}")
print(f"Fold Scores:            {[round(s, 4) for s in fold_scores]}")
print(f"Delta vs Official SVC (0.3749): {mean_acc - 0.3749:+.4f} ({(mean_acc - 0.3749)*100:+.2f}%)")
print(f"Total Runtime:          {total_elapsed:.1f}s")
print("=" * 75)

In [ ]:
out_path_oof = os.path.join(OUTPUTS_DIR, "oof_open_pool_svc.npy")
np.save(out_path_oof, oof_probs)
print(f"Saved OOF probabilities to {out_path_oof}")

In [ ]:
report_path = os.path.join(REPORTS_DIR, "open_pool_svc_results.md")
cls_rep = classification_report(y_true_idx, oof_preds, target_names=classes, output_dict=True)
with open(report_path, "w") as f:
    f.write(f"""# EXP_28: Official + Cleaned Open Pool Multi-View LinearSVC Report
**Experiment ID**: `EXP_28_open_pool_svc`  
**Date**: {datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")}  
**Validation Strategy**: 5-Fold `GroupKFold(group=prompt_id)` (Strict Prompt-Disjoint)  
**Training Data**: Official Data (2,159 rows, w=1.0) + Clean Open Pool (~36,000 rows, w=0.35)  
**Primary Metric**: Balanced Accuracy = **{mean_acc:.4f}** (Macro F1 = **{macro_f1:.4f}**)  
**Fold Scores**: {[round(s, 4) for s in fold_scores]} (Std: ±{np.std(fold_scores):.4f})  
**Runtime**: {total_elapsed:.1f}s  
**Contamination Filtering**: Purged an average of {np.mean(contamination_stats):.1f} near-prompt rows per fold.  

---

## 1. Executive Summary & Impact Analysis
- **Official 4-View SVC Baseline (`EXP_27`)**: `0.3749` Balanced Accuracy
- **Official + Clean Open Pool 4-View SVC (`EXP_28`)**: **`{mean_acc:.4f}`** Balanced Accuracy
- **Net Delta**: **`{mean_acc - 0.3749:+.4f}`** (`{(mean_acc - 0.3749)*100:+.2f}%` relative gain)

## 2. Per-Class Recall Forensics
| Model Class | Recall | F1-Score |
| :--- | :---: | :---: |
""" + "\n".join([f"| `{c}` | {cls_rep[c]['recall']:.2%} | {cls_rep[c]['f1-score']:.4f} |" for c in classes]) + f"""
""")

print(f"Report written to {report_path}")

In [ ]:
log_path = os.path.join(EXPERIMENTS_DIR, "experiment_log.jsonl")
record = {
    "experiment_id": "28_open_pool_svc",
    "date": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "research_question": "Does adding 36,000 clean Open Pool rows improve prompt-disjoint balanced accuracy over official-only SVC?",
    "hypothesis": "Expanding 4-view n-gram vocabulary with 36,000 balanced external passages (w=0.35) improves generalization on unseen prompts",
    "research_reference": "Open Pool v11 (crtproduction) + Ertuğrul Özer 0.672",
    "dataset_version": "Official v1 + Open Pool Clean (36,000)",
    "feature_method": "4-View TF-IDF (Char 3-5, Word 1-3, Shape 2-6, Function 1-3)",
    "model": "LinearSVC(C=0.5, class_weight='balanced', sample_weight=[1.0, 0.35])",
    "hyperparameters": {"C": 0.5, "ext_rows": 36000, "ext_weight": 0.35, "views": 4},
    "cv_strategy": "GroupKFold(n_splits=5, group=prompt_id) (Strict Prompt-Disjoint)",
    "group_column": "prompt_id",
    "random_seed": 42,
    "fold_scores": [float(s) for s in fold_scores],
    "mean_balanced_accuracy": float(mean_acc),
    "std_balanced_accuracy": float(np.std(fold_scores)),
    "macro_f1": float(macro_f1),
    "per_class_recall": {c: float(cls_rep[c]["recall"]) for c in classes},
    "training_time_sec": round(total_elapsed, 2),
    "status": "completed",
    "conclusion": f"Open Pool SVC achieved {mean_acc:.4f} Balanced Accuracy ({mean_acc - 0.3749:+.4f} over official-only SVC 0.3749).",
    "next_recommendation": "EXP_29: Official + Open Pool + Prompt Residuals",
}
with open(log_path, "a") as f:
    f.write(json.dumps(record) + "\n")

print("Registered EXP_28_open_pool_svc in experiment_log.jsonl")